# Aşama 13 — Statistical & Error Analysis
Kaydedilmiş final-test tahminlerinden per-class F1, 3-seed normalize confusion matrix ve paired stratified bootstrap %95 CI üretir. **Model inference/eğitim yapmaz.**


In [3]:
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.metrics import f1_score,precision_recall_fscore_support,confusion_matrix,ConfusionMatrixDisplay
R=Path.cwd();I=R/"outputs/final_test_v2";O=R/"outputs/asama13_analysis";O.mkdir(parents=True,exist_ok=True)
C=sorted(p.name for p in (R/"wbc_project/data/processed/phase2_test").iterdir() if p.is_dir());M=["convnext_tiny","resnet50","retfound_fullft"];S=[42,1,7]
D={};ref=None
for m in M:
 for s in S:
  d=I/m/f"seed{s}";y=np.load(d/"true.npy");p=np.load(d/"pred.npy");D[m,s]=(y,p)
  if ref is None:ref=y
  assert np.array_equal(y,ref)
rows=[]
for m in M:
 for s in S:
  y,p=D[m,s];pr,re,f1,sup=precision_recall_fscore_support(y,p,labels=np.arange(13),zero_division=0)
  for i,c in enumerate(C):rows.append([m,s,c,pr[i],re[i],f1[i],sup[i]])
df=pd.DataFrame(rows,columns=["model","seed","class","precision","recall","f1","support"]);df.to_csv(O/"per_class_all_seeds.csv",index=False)
sm=df.groupby(["model","class"]).agg(f1_mean=("f1","mean"),f1_sd=("f1","std"),recall_mean=("recall","mean"),precision_mean=("precision","mean"),support=("support","first")).reset_index();sm.to_csv(O/"per_class_summary.csv",index=False)
for m in M:
 cm=sum(confusion_matrix(D[m,s][0],D[m,s][1],labels=np.arange(13)) for s in S);norm=cm/cm.sum(1,keepdims=True);np.save(O/f"{m}_cm_counts.npy",cm)
 fig,ax=plt.subplots(figsize=(11,9));ConfusionMatrixDisplay(norm,display_labels=C).plot(ax=ax,xticks_rotation=45,values_format=".2f",colorbar=False);ax.set_title(f"{m} — normalized test confusion matrix (3 seeds)");fig.tight_layout();fig.savefig(O/f"{m}_confusion_matrix.png",dpi=300,bbox_inches="tight");plt.close(fig)
rng=np.random.default_rng(2026);idxs=[np.flatnonzero(ref==i) for i in range(13)];pairs=[("resnet50","convnext_tiny"),("resnet50","retfound_fullft"),("convnext_tiny","retfound_fullft")];br=[]
for a,b in pairs:
 vals=[]
 for s in S:
  pa=D[a,s][1];pb=D[b,s][1]
  for _ in range(500):
   ix=np.concatenate([rng.choice(q,len(q),replace=True) for q in idxs]);vals.append(f1_score(ref[ix],pa[ix],average="macro",zero_division=0)-f1_score(ref[ix],pb[ix],average="macro",zero_division=0))
 point=np.mean([f1_score(ref,D[a,s][1],average="macro")-f1_score(ref,D[b,s][1],average="macro") for s in S]);lo,hi=np.quantile(vals,[.025,.975]);br.append([f"{a} - {b}",point,lo,hi]);print(f"{a} - {b}: Δ={point:+.4f}, 95% CI [{lo:+.4f},{hi:+.4f}]")
pd.DataFrame(br,columns=["comparison","delta_macro_f1","ci_low","ci_high"]).to_csv(O/"bootstrap_macro_f1.csv",index=False)
print("\n=== PER-CLASS F1 mean ± SD ===")
for c in C:
 print(c," | "," | ".join(f"{m}: {sm[(sm.model==m)&(sm['class']==c)].iloc[0].f1_mean:.3f}±{sm[(sm.model==m)&(sm['class']==c)].iloc[0].f1_sd:.3f}" for m in M))
print("\n✓ Kaydedildi:",O.resolve())


resnet50 - convnext_tiny: Δ=+0.0099, 95% CI [-0.0242,+0.0397]
resnet50 - retfound_fullft: Δ=+0.0359, 95% CI [+0.0065,+0.0589]
convnext_tiny - retfound_fullft: Δ=+0.0260, 95% CI [+0.0077,+0.0415]

=== PER-CLASS F1 mean ± SD ===
BA  |  convnext_tiny: 0.949±0.014 | resnet50: 0.954±0.006 | retfound_fullft: 0.948±0.006
BL  |  convnext_tiny: 0.882±0.018 | resnet50: 0.893±0.007 | retfound_fullft: 0.877±0.008
BNE  |  convnext_tiny: 0.388±0.055 | resnet50: 0.264±0.111 | retfound_fullft: 0.240±0.005
EO  |  convnext_tiny: 0.961±0.005 | resnet50: 0.960±0.006 | retfound_fullft: 0.956±0.007
LY  |  convnext_tiny: 0.964±0.003 | resnet50: 0.964±0.005 | retfound_fullft: 0.958±0.002
MMY  |  convnext_tiny: 0.505±0.029 | resnet50: 0.513±0.008 | retfound_fullft: 0.404±0.059
MO  |  convnext_tiny: 0.936±0.007 | resnet50: 0.929±0.010 | retfound_fullft: 0.909±0.005
MY  |  convnext_tiny: 0.716±0.050 | resnet50: 0.754±0.004 | retfound_fullft: 0.589±0.050
PC  |  convnext_tiny: 0.000±0.000 | resnet50: 0.027±0.047 |

In [2]:
pip install pandas

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   ------- -------------------------------- 1.8/9.9 MB 7.2 MB/s eta 0:00:02
   --------------- ------------------------ 3.9/9.9 MB 8.7 MB/s eta 0:00:01
   -------------------------- ------------- 6.6/9.9 MB 9.8 MB/s eta 0:00:01
   ------------------------------- -------- 7.9/9.9 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 9.7 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.
